# Method of Lines and Runge-Kutta

Connect spatial right-hand sides to Runge-Kutta time stages using NRPy time-stepping utilities.

Navigation: [Index](../index.ipynb) | Previous: [Wave Equation with NumPy](wave_equation_with_numpy.ipynb) | Next: [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)

## Why This Matters
The Method of Lines turns a spatially discretized PDE into ordinary time updates. Runge-Kutta methods specify the stages of those updates.

## What You Will See
You will inspect printed expressions, generated artifacts, or diagnostic tables and then connect them to the next notebook.

Prerequisite bridge: this notebook follows [Wave Equation with NumPy](wave_equation_with_numpy.ipynb). Terms are defined here before they are used.

## RK4 Stages and NRPy Registration
The right-hand side computes time derivatives. RK4 combines four stage evaluations into one time step.

In [1]:
import nrpy.c_function as cfc
from nrpy.infrastructures import BHaH
from nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time import register_CFunction_MoL_step_forward_in_time
from nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary import (
    generate_Butcher_tables,
    intermediate_stage_gf_names_list,
    is_diagonal_Butcher,
    validate,
)

butcher_tables = generate_Butcher_tables()
validate(butcher_tables, ["RK4"], verbose=False)
rk4_table, rk4_order = butcher_tables["RK4"]
print("RK4 order:", rk4_order)
print("is diagonal:", is_diagonal_Butcher(butcher_tables, "RK4"))
print("intermediate storage names:", intermediate_stage_gf_names_list(butcher_tables, "RK4"))

cfc.CFunction_dict.pop("MoL_step_forward_in_time", None)
for name in ["RK_STEP1", "RK_STEP2", "RK_STEP3", "RK_STEP4"]:
    BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.pop(name, None)
register_CFunction_MoL_step_forward_in_time(
    butcher_tables,
    "RK4",
    rhs_string="rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);",
)
registered = cfc.CFunction_dict["MoL_step_forward_in_time"]
print("registered MoL function names:", sorted(name for name in cfc.CFunction_dict if "MoL" in name or "mol" in name))
print("complete prototype:")
print(registered.function_prototype)
print("registered substep names:", sorted(BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict))

full_function = registered.full_function
start = full_function.index("static void rk_substep_1_host")
end_marker = "} // END FUNCTION: rk_substep_1_host"
end = full_function.index(end_marker, start) + len(end_marker)
print("complete generated RK substep block:")
print(full_function[start:end])

RK4: PASSED VALIDATION
RK4 order: 4
is diagonal: True
intermediate storage names: ['y_nplus1_running_total_gfs', 'k_odd_gfs', 'k_even_gfs']
registered MoL function names: ['MoL_step_forward_in_time']
complete prototype:
void MoL_step_forward_in_time(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
registered substep names: ['RK_STEP1', 'RK_STEP2', 'RK_STEP3', 'RK_STEP4']
complete generated RK substep block:
static void rk_substep_1_host(params_struct *restrict params, REAL *restrict k_odd_gfs, REAL *restrict y_n_gfs,
                              REAL *restrict y_nplus1_running_total_gfs, const REAL dt) {
  LOOP_ALL_GFS_GPS(i) {
    const REAL k_odd_gfsL = k_odd_gfs[i];
    const REAL y_n_gfsL = y_n_gfs[i];
    const REAL RK_Rational_1_6 = 1.0 / 6.0;
    const REAL RK_Rational_1_2 = 1.0 / 2.0;
    y_nplus1_running_total_gfs[i] = RK_Rational_1_6 * dt * k_odd_gfsL;
    k_odd_gfs[i] = RK_Rational_1_2 * dt * k_odd_gfsL + y_n_gfsL;
  }
} // END FUNCTION: rk_subst

## Interpreting the Output
The RK4 validation confirms the stage table, and the generated substep block shows how NRPy turns those stages into C. This is the time-integration driver used by generated wave projects.

## Where This Leads
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)